# Metal Surface Defect Dataset → Azure Custom Vision
Dataset: [SciDB – Metal Surface Defect Dataset](https://www.scidb.cn/en/detail?dataSetId=3d739ddb4bdc439a9bf7ef550cae48d8)

We don't yet know the dataset's exact format, so this notebook **discovers it first** and then:

1. Loads the data (Drive / upload / URL)
2. **Inventories** files and auto-detects the annotation format (Pascal VOC XML, YOLO txt, COCO JSON, segmentation masks, or class-per-folder)
3. Parses everything into **one unified table**
4. **EDA**: class balance, box sizes, image sizes, sample visualisations
5. **Quality checks**: corrupt files, duplicates, invalid/out-of-bounds boxes, unlabeled images
6. **Preprocessing** for Custom Vision limits (RGB JPEG, ≤6 MB, short side ≥256 px, normalised boxes)
7. Stratified **train / holdout split** (the holdout set is never uploaded, so we can test on it ourselves)
8. **Upload → train → test** with the Azure Custom Vision SDK (optional; you need Azure keys)

Run the cells top to bottom. Wherever there is a `# >>> EDIT` marker, check the setting before running.

## 0. Setup

In [ ]:
!pip -q install azure-cognitiveservices-vision-customvision msrest imagehash pyyaml tqdm

import os, re, json, glob, shutil, hashlib, zipfile, random, xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image, ImageOps
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
WORK = Path('/content/work'); RAW = WORK/'raw'; OUT = WORK/'customvision'
for p in (RAW, OUT): p.mkdir(parents=True, exist_ok=True)
IMG_EXT = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.gif','.webp'}

## 1. Get the data
SciDB usually requires you to download in a browser (you may need to log in). The easiest route:
**download the archive(s) on your computer → put them in Google Drive → set `DRIVE_PATH`.**
Alternatives: set `DIRECT_URL` if you have a direct link, or use `files.upload()`.

In [ ]:
# >>> EDIT: choose one source
SOURCE      = 'drive'            # 'drive' | 'url' | 'upload'
DRIVE_PATH  = '/content/drive/MyDrive/datasets/metal_surface_defect'  # folder or .zip/.rar/.7z/.tar
DIRECT_URL  = ''

def extract_all(folder: Path):
    # Extract every archive in the folder, including nested ones, until none are left
    done = set()
    while True:
        arcs = [p for p in folder.rglob('*') if p.suffix.lower() in {'.zip','.rar','.7z','.tar','.gz','.tgz'} and p not in done]
        if not arcs: break
        for a in arcs:
            dest = a.with_suffix('')  if a.suffix != '.gz' else a.parent/a.stem.replace('.tar','')
            dest.mkdir(exist_ok=True)
            print('extracting', a.name)
            if a.suffix.lower() == '.zip': zipfile.ZipFile(a).extractall(dest)
            elif a.suffix.lower() in {'.rar','.7z'}: os.system(f'7z x -y -o"{dest}" "{a}" > /dev/null')
            else: shutil.unpack_archive(str(a), str(dest))
            done.add(a)

if SOURCE == 'drive':
    from google.colab import drive; drive.mount('/content/drive')
    src = Path(DRIVE_PATH)
    if src.is_dir(): shutil.copytree(src, RAW/src.name, dirs_exist_ok=True)
    else: shutil.copy(src, RAW)
elif SOURCE == 'url':
    !wget -q --show-progress -P {RAW} "{DIRECT_URL}"
else:
    from google.colab import files
    for name, data in files.upload().items(): (RAW/name).write_bytes(data)

extract_all(RAW)
print('done ->', RAW)

## 2. Inventory: what is in the dataset?

In [ ]:
all_files = [p for p in RAW.rglob('*') if p.is_file() and '__MACOSX' not in p.parts and not p.name.startswith('.')]
ext_counts = Counter(p.suffix.lower() for p in all_files)
print('Total files:', len(all_files)); print(pd.Series(ext_counts).sort_values(ascending=False).to_string())

def tree(root, max_depth=3, max_items=6, _d=0):
    if _d > max_depth: return
    entries = sorted(root.iterdir()); dirs=[e for e in entries if e.is_dir()]; fs=[e for e in entries if e.is_file()]
    for d in dirs[:max_items]:
        n = sum(1 for _ in d.rglob('*') if _.is_file())
        print('  '*_d + f'[{d.name}/]  ({n} files)'); tree(d, max_depth, max_items, _d+1)
    if len(dirs) > max_items: print('  '*_d + f'... +{len(dirs)-max_items} more dirs')
    for f in fs[:3]: print('  '*_d + f.name)
    if len(fs) > 3: print('  '*_d + f'... +{len(fs)-3} more files')
tree(RAW)

# Print a peek at a few non-image files, e.g. labels and READMEs
for p in [p for p in all_files if p.suffix.lower() in {'.xml','.txt','.json','.yaml','.yml','.csv','.md'}][:5]:
    print('\n=====', p.relative_to(RAW)); print(p.read_text(errors='ignore')[:600])

In [ ]:
images = [p for p in all_files if p.suffix.lower() in IMG_EXT]
mask_like = [p for p in images if re.search(r'(mask|label|gt|annotation|seg)', str(p).lower())]
xmls  = [p for p in all_files if p.suffix.lower()=='.xml']
txts  = [p for p in all_files if p.suffix.lower()=='.txt' and p.name.lower() not in {'classes.txt','readme.txt'}]
jsons = [p for p in all_files if p.suffix.lower()=='.json']
coco_jsons = []
for j in jsons:
    try:
        d = json.loads(j.read_text(errors='ignore'))
        if isinstance(d, dict) and {'images','annotations'} <= d.keys(): coco_jsons.append(j)
    except Exception: pass
yolo_txts = [t for t in txts[:200] if re.match(r'^\s*\d+\s+[\d.]+\s+[\d.]+\s+[\d.]+\s+[\d.]+', t.read_text(errors='ignore'))]

scores = {'voc': len(xmls), 'coco': 10**6 if coco_jsons else 0, 'yolo': len(yolo_txts)*(len(txts)/max(1,min(len(txts),200))),
          'mask': len(mask_like), 'folders': 0}
FORMAT = max(scores, key=scores.get) if max(scores.values()) > 0 else 'folders'
print('Images:', len(images), '| xml:', len(xmls), '| yolo txt:', len(yolo_txts), '| coco json:', len(coco_jsons), '| mask-like images:', len(mask_like))
print('>>> Detected annotation format:', FORMAT)
# >>> EDIT: override if detection is wrong, e.g. FORMAT = 'voc'

## 3. Parse into a unified table
Every row is **one box** (or one image-level label when the format is `folders`):
`image, width, height, label, xmin, ymin, xmax, ymax` (pixels).

In [ ]:
def img_size(p):
    with Image.open(p) as im: return im.size

def stem_index(paths):
    idx = defaultdict(list)
    for p in paths: idx[p.stem].append(p)
    return idx
img_by_stem = stem_index([p for p in images if p not in set(mask_like)])

rows = []
if FORMAT == 'voc':
    for x in tqdm(xmls):
        r = ET.parse(x).getroot()
        fn = (r.findtext('filename') or x.stem)
        cand = img_by_stem.get(Path(fn).stem) or img_by_stem.get(x.stem)
        if not cand: continue
        ip = cand[0]; W, H = img_size(ip)
        objs = r.findall('object')
        if not objs: rows.append(dict(image=ip, width=W, height=H, label=None))
        for o in objs:
            b = o.find('bndbox')
            rows.append(dict(image=ip, width=W, height=H, label=o.findtext('name').strip(),
                             xmin=float(b.findtext('xmin')), ymin=float(b.findtext('ymin')),
                             xmax=float(b.findtext('xmax')), ymax=float(b.findtext('ymax'))))

elif FORMAT == 'yolo':
    names = None
    for y in [p for p in all_files if p.suffix in {'.yaml','.yml'}]:
        import yaml; d = yaml.safe_load(y.read_text()); n = d.get('names') if isinstance(d, dict) else None
        if n: names = n if isinstance(n, list) else [n[k] for k in sorted(n)]
    ct = [p for p in all_files if p.name.lower()=='classes.txt']
    if names is None and ct: names = [l.strip() for l in ct[0].read_text().splitlines() if l.strip()]
    print('class names:', names)
    for t in tqdm(txts):
        cand = img_by_stem.get(t.stem)
        if not cand: continue
        ip = cand[0]; W, H = img_size(ip); lines = [l.split() for l in t.read_text().splitlines() if l.strip()]
        if not lines: rows.append(dict(image=ip, width=W, height=H, label=None))
        for c, cx, cy, w, h in (l[:5] for l in lines):
            cx, cy, w, h = map(float, (cx, cy, w, h)); c = int(c)
            rows.append(dict(image=ip, width=W, height=H, label=names[c] if names else str(c),
                             xmin=(cx-w/2)*W, ymin=(cy-h/2)*H, xmax=(cx+w/2)*W, ymax=(cy+h/2)*H))

elif FORMAT == 'coco':
    for j in coco_jsons:
        d = json.loads(j.read_text()); cats = {c['id']: c['name'] for c in d['categories']}
        by_img = defaultdict(list)
        for a in d['annotations']: by_img[a['image_id']].append(a)
        for im in d['images']:
            cand = img_by_stem.get(Path(im['file_name']).stem)
            if not cand: continue
            ip = cand[0]; W, H = im.get('width') or img_size(ip)[0], im.get('height') or img_size(ip)[1]
            if not by_img[im['id']]: rows.append(dict(image=ip, width=W, height=H, label=None))
            for a in by_img[im['id']]:
                x, y, w, h = a['bbox']
                rows.append(dict(image=ip, width=W, height=H, label=cats[a['category_id']], xmin=x, ymin=y, xmax=x+w, ymax=y+h))

elif FORMAT == 'mask':
    # Masks -> boxes via connected components. The class comes from the mask pixel value, or from the parent folder name if the mask is binary.
    import cv2
    for m in tqdm(mask_like):
        cand = [p for p in img_by_stem.get(m.stem, []) if p != m] or img_by_stem.get(re.sub(r'(_mask|_label|_gt)$', '', m.stem), [])
        if not cand: continue
        ip = cand[0]; arr = np.array(Image.open(m).convert('L')); H, W = arr.shape
        vals = [v for v in np.unique(arr) if v != 0]
        if not vals: rows.append(dict(image=ip, width=W, height=H, label=None))
        for v in vals:
            n, lab, stats, _ = cv2.connectedComponentsWithStats((arr == v).astype(np.uint8))
            for k in range(1, n):
                x, y, w, h, area = stats[k]
                if area < 20: continue  # speckle noise
                lbl = ip.parent.name if len(vals) == 1 else f'class_{v}'
                rows.append(dict(image=ip, width=W, height=H, label=lbl, xmin=x, ymin=y, xmax=x+w, ymax=y+h))

else:  # 'folders': the class name is the parent folder (classification only)
    for ip in tqdm(images):
        W, H = img_size(ip); rows.append(dict(image=ip, width=W, height=H, label=ip.parent.name))

df = pd.DataFrame(rows)
TASK = 'classification' if FORMAT == 'folders' else 'detection'
print(TASK, '| rows:', len(df), '| images:', df.image.nunique()); df.head()

## 4. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
df.label.fillna('<no defect>').value_counts().plot.bar(ax=ax[0], title='Instances per class')
df.groupby('image').label.count().value_counts().sort_index().plot.bar(ax=ax[1], title='Labels per image')
sz = df.drop_duplicates('image')[['width','height']].value_counts().head(10)
sz.plot.barh(ax=ax[2], title='Top image sizes (W,H)'); plt.tight_layout(); plt.show()

if TASK == 'detection':
    b = df.dropna(subset=['xmin']).copy()
    b['bw'] = (b.xmax-b.xmin)/b.width; b['bh'] = (b.ymax-b.ymin)/b.height; b['area'] = b.bw*b.bh
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    for l, g in b.groupby('label'): ax[0].scatter(g.bw, g.bh, s=4, alpha=.4, label=l)
    ax[0].set(xlabel='box w / img w', ylabel='box h / img h', title='Relative box size'); ax[0].legend(markerscale=4)
    b.boxplot('area', by='label', ax=ax[1], rot=45); ax[1].set_yscale('log'); plt.suptitle(''); plt.show()
    print(b.groupby('label')[['bw','bh','area']].describe().round(3).T)

In [ ]:
def show_samples(df, per_class=3, cols=6):
    picks = []
    for l, g in df.dropna(subset=['label']).groupby('label'):
        picks += list(g.image.drop_duplicates().sample(min(per_class, g.image.nunique()), random_state=SEED))
    rows_ = int(np.ceil(len(picks)/cols)); fig, axs = plt.subplots(rows_, cols, figsize=(3*cols, 3*rows_))
    colors = {l: plt.cm.tab10(i % 10) for i, l in enumerate(sorted(df.label.dropna().unique()))}
    for a, ip in zip(np.ravel(axs), picks):
        a.imshow(Image.open(ip).convert('RGB')); a.set_title(ip.name[:25], fontsize=8)
        for _, r in df[df.image == ip].iterrows():
            if TASK == 'detection' and pd.notna(r.get('xmin')):
                a.add_patch(patches.Rectangle((r.xmin, r.ymin), r.xmax-r.xmin, r.ymax-r.ymin, fill=False, ec=colors[r.label], lw=1.5))
                a.text(r.xmin, r.ymin, r.label, color='w', fontsize=6, backgroundcolor=colors[r.label])
            elif r.label: a.set_xlabel(r.label)
    for a in np.ravel(axs): a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()
show_samples(df)

## 5. Quality checks and cleaning

In [ ]:
issues = defaultdict(list)
# 5a. Corrupt or unreadable images
for ip in tqdm(df.image.unique(), desc='verify'):
    try:
        with Image.open(ip) as im: im.verify()
    except Exception as e: issues['corrupt'].append(ip)

# 5b. Exact duplicate images (md5), possibly with conflicting labels
h = {ip: hashlib.md5(Path(ip).read_bytes()).hexdigest() for ip in df.image.unique()}
df['md5'] = df.image.map(h)
dups = df.drop_duplicates('image').groupby('md5').image.apply(list); dups = dups[dups.str.len() > 1]
issues['duplicate_groups'] = list(dups)

# 5c. Box problems
if TASK == 'detection':
    bx = df.dropna(subset=['xmin'])
    issues['out_of_bounds'] = bx[(bx.xmin < 0)|(bx.ymin < 0)|(bx.xmax > bx.width)|(bx.ymax > bx.height)].index.tolist()
    issues['degenerate']    = bx[(bx.xmax-bx.xmin < 2)|(bx.ymax-bx.ymin < 2)].index.tolist()
    issues['exact_dup_boxes'] = bx[bx.duplicated(['image','label','xmin','ymin','xmax','ymax'])].index.tolist()

# 5d. Images with no annotation file at all
labelled = set(df.image); issues['images_without_labels'] = [p for p in images if p not in labelled and p not in set(mask_like)]
for k, v in issues.items(): print(f'{k:24s}: {len(v)}')

In [ ]:
# Fix what can be fixed automatically. Review the counts above first.
clean = df[~df.image.isin(issues['corrupt'])].copy()
if TASK == 'detection':
    m = clean.xmin.notna()
    clean.loc[m, 'xmin'] = clean.loc[m, 'xmin'].clip(lower=0); clean.loc[m, 'ymin'] = clean.loc[m, 'ymin'].clip(lower=0)
    clean.loc[m, 'xmax'] = np.minimum(clean.loc[m, 'xmax'], clean.loc[m, 'width'])
    clean.loc[m, 'ymax'] = np.minimum(clean.loc[m, 'ymax'], clean.loc[m, 'height'])
    clean = clean.drop(index=[i for i in issues['degenerate'] + issues['exact_dup_boxes'] if i in clean.index])
# Keep one copy of each duplicate image
keep = clean.drop_duplicates('md5').image
clean = clean[clean.image.isin(keep)]

# >>> EDIT: optional class renaming or merging, e.g. {'scratches': 'scratch'}
CLASS_MAP = {}
clean['label'] = clean.label.replace(CLASS_MAP)
# >>> EDIT: drop classes with too few examples (Custom Vision needs >=15 images per tag for detection, >=5 for classification)
MIN_IMAGES_PER_TAG = 15 if TASK == 'detection' else 5
per_tag = clean.dropna(subset=['label']).groupby('label').image.nunique(); print(per_tag)
rare = per_tag[per_tag < MIN_IMAGES_PER_TAG].index.tolist()
if rare: print('Dropping rare tags:', rare); clean = clean[~clean.label.isin(rare)]
print('clean rows', len(clean), '| images', clean.image.nunique())

## 6. Train / holdout split (stratified)
Custom Vision does its own internal split for its training metrics. We **also hold out** a test set that is never uploaded, so we get an honest evaluation in section 9.

In [ ]:
HOLDOUT_FRAC = 0.15   # >>> EDIT
img_tbl = clean.groupby('image').label.agg(lambda s: Counter(s.dropna()).most_common(1)[0][0] if s.notna().any() else '<none>')
from sklearn.model_selection import train_test_split
tr_imgs, ho_imgs = train_test_split(img_tbl.index, test_size=HOLDOUT_FRAC, stratify=img_tbl.values, random_state=SEED)
clean['split'] = np.where(clean.image.isin(set(ho_imgs)), 'holdout', 'train')
print(clean.drop_duplicates('image').groupby(['split']).size())
print(pd.crosstab(clean.label, clean.split))

## 7. Preprocess and export
Custom Vision limits (as documented by Azure): training images **≤ 6 MB**, short side **≥ 256 px** recommended, and regions as **normalised** `left, top, width, height` in the 0–1 range.
We convert to RGB JPEG. Grayscale steel images are converted to 3 channels, and any image smaller than 256 px is upscaled. Because the boxes are normalised, upscaling doesn't change them.

In [ ]:
MIN_SIDE, MAX_BYTES = 256, 6 * 1024 * 1024
IMG_OUT = OUT/'images'; IMG_OUT.mkdir(exist_ok=True)

def prep(ip, dst):
    im = ImageOps.exif_transpose(Image.open(ip)).convert('RGB')
    s = min(im.size)
    if s < MIN_SIDE: im = im.resize((round(im.width*MIN_SIDE/s), round(im.height*MIN_SIDE/s)), Image.BICUBIC)
    q = 95
    while True:
        im.save(dst, 'JPEG', quality=q)
        if dst.stat().st_size <= MAX_BYTES or q <= 50: break
        q -= 10
    return dst

manifest = []
for i, (ip, g) in enumerate(tqdm(clean.groupby('image'), desc='export')):
    r0 = g.iloc[0]; name = f'{i:05d}_{Path(ip).stem}.jpg'; prep(ip, IMG_OUT/name)
    regions = []
    if TASK == 'detection':
        for _, r in g.dropna(subset=['xmin']).iterrows():
            regions.append(dict(tag=r.label, left=r.xmin/r.width, top=r.ymin/r.height,
                                width=(r.xmax-r.xmin)/r.width, height=(r.ymax-r.ymin)/r.height))
    manifest.append(dict(file=name, split=r0.split, source=str(Path(ip).relative_to(RAW)),
                         tags=sorted(set(g.label.dropna())), regions=regions))
(OUT/'manifest.json').write_text(json.dumps(manifest, indent=1))
print(len(manifest), 'images exported ->', OUT)

In [ ]:
# Sanity check: re-draw a few exported images using only the normalised regions from the manifest
if TASK == 'detection':
    fig, axs = plt.subplots(1, 4, figsize=(16, 4))
    for a, m in zip(axs, random.sample([m for m in manifest if m['regions']], 4)):
        im = Image.open(IMG_OUT/m['file']); a.imshow(im); W, H = im.size
        for r in m['regions']:
            a.add_patch(patches.Rectangle((r['left']*W, r['top']*H), r['width']*W, r['height']*H, fill=False, ec='lime'))
        a.set_title(', '.join(m['tags']), fontsize=8); a.axis('off')
    plt.show()

# Zip and save to Drive so you don't have to redo this
shutil.make_archive(str(WORK/'metal_defect_customvision'), 'zip', OUT)
if Path('/content/drive/MyDrive').exists():
    shutil.copy(WORK/'metal_defect_customvision.zip', '/content/drive/MyDrive/'); print('saved to Drive')

## 8. Upload and train on Azure Custom Vision (optional)
Create a **Custom Vision** resource in the Azure portal. Use a training resource and a prediction resource, or a single multi-service resource.
Put the keys in Colab **Secrets** (the 🔑 icon): `CV_TRAINING_KEY`, `CV_TRAINING_ENDPOINT`, `CV_PREDICTION_KEY`, `CV_PREDICTION_RESOURCE_ID`, `CV_PREDICTION_ENDPOINT`.

In [ ]:
RUN_AZURE = False   # >>> EDIT: set True once your keys are in Colab Secrets
if RUN_AZURE:
    from google.colab import userdata
    from azure.cognitiveservices.vision.customvision.training import CustomVisionTrainingClient
    from azure.cognitiveservices.vision.customvision.training.models import (
        ImageFileCreateBatch, ImageFileCreateEntry, Region)
    from msrest.authentication import ApiKeyCredentials
    import time

    trainer = CustomVisionTrainingClient(userdata.get('CV_TRAINING_ENDPOINT'),
                                         ApiKeyCredentials(in_headers={'Training-key': userdata.get('CV_TRAINING_KEY')}))
    domain_type = 'ObjectDetection' if TASK == 'detection' else 'Classification'
    # 'General' is a good default. Use the '(compact)' domain if you need to export the model for edge devices.
    domain = next(d for d in trainer.get_domains() if d.type == domain_type and d.name == 'General')
    kw = {} if TASK == 'detection' else {'classification_type': 'Multilabel' if any(len(m['tags']) > 1 for m in manifest) else 'Multiclass'}
    project = trainer.create_project('metal-surface-defects', domain_id=domain.id, **kw)
    tags = {t: trainer.create_tag(project.id, t) for t in sorted({t for m in manifest for t in m['tags']})}

    train_items = [m for m in manifest if m['split'] == 'train' and m['tags']]
    for i in tqdm(range(0, len(train_items), 64), desc='upload'):   # the API accepts at most 64 images per batch
        entries = []
        for m in train_items[i:i+64]:
            data = (IMG_OUT/m['file']).read_bytes()
            if TASK == 'detection':
                regs = [Region(tag_id=tags[r['tag']].id, left=r['left'], top=r['top'], width=r['width'], height=r['height']) for r in m['regions']]
                entries.append(ImageFileCreateEntry(name=m['file'], contents=data, regions=regs))
            else:
                entries.append(ImageFileCreateEntry(name=m['file'], contents=data, tag_ids=[tags[t].id for t in m['tags']]))
        res = trainer.create_images_from_files(project.id, ImageFileCreateBatch(images=entries))
        if not res.is_batch_successful:
            bad = [im for im in res.images if im.status not in ('OK', 'OKDuplicate')]
            print('batch issues:', [(b.source_url, b.status) for b in bad][:5])

    iteration = trainer.train_project(project.id)   # quick training. For advanced training: training_type='Advanced', reserved_budget_in_hours=1
    while iteration.status != 'Completed':
        time.sleep(30); iteration = trainer.get_iteration(project.id, iteration.id); print('training:', iteration.status)
    perf = trainer.get_iteration_performance(project.id, iteration.id, threshold=0.5, overlap_threshold=0.3)
    print(f'Azure internal metrics  P={perf.precision:.3f}  R={perf.recall:.3f}  mAP={perf.average_precision:.3f}')
    for t in perf.per_tag_performance: print(f'  {t.name:20s} P={t.precision:.3f} R={t.recall:.3f} AP={t.average_precision:.3f}')
    azure_metrics = dict(precision=perf.precision, recall=perf.recall, mAP=perf.average_precision,
                         per_class={t.name: dict(precision=t.precision, recall=t.recall, AP=t.average_precision)
                                    for t in perf.per_tag_performance})

    PUBLISH_NAME = 'metal-defects-v1'
    trainer.publish_iteration(project.id, iteration.id, PUBLISH_NAME, userdata.get('CV_PREDICTION_RESOURCE_ID'))

## 9. Test on our own holdout set
This computes per-class **precision, recall and AP@0.5 IoU** (detection) or accuracy (classification) on images that Azure never saw during training.

In [ ]:
def iou(a, b):
    ax2, ay2, bx2, by2 = a[0]+a[2], a[1]+a[3], b[0]+b[2], b[1]+b[3]
    iw = max(0, min(ax2, bx2)-max(a[0], b[0])); ih = max(0, min(ay2, by2)-max(a[1], b[1])); inter = iw*ih
    return inter / (a[2]*a[3] + b[2]*b[3] - inter + 1e-9)

def ap_voc(rec, prec):
    mrec = np.concatenate(([0], rec, [1])); mpre = np.concatenate(([0], prec, [0]))
    for i in range(len(mpre)-2, -1, -1): mpre[i] = max(mpre[i], mpre[i+1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]; return float(np.sum((mrec[idx+1]-mrec[idx]) * mpre[idx+1]))

if RUN_AZURE:
    from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
    predictor = CustomVisionPredictionClient(userdata.get('CV_PREDICTION_ENDPOINT'),
                                             ApiKeyCredentials(in_headers={'Prediction-key': userdata.get('CV_PREDICTION_KEY')}))
    hold = [m for m in manifest if m['split'] == 'holdout']; preds = {}
    for m in tqdm(hold, desc='predict'):
        with open(IMG_OUT/m['file'], 'rb') as f:
            fn = predictor.detect_image if TASK == 'detection' else predictor.classify_image
            preds[m['file']] = fn(project.id, PUBLISH_NAME, f).predictions
        time.sleep(0.5)   # stay under the free-tier rate limit

    if TASK == 'detection':
        results = []
        for cls in tags:
            gts = {m['file']: [r for r in m['regions'] if r['tag'] == cls] for m in hold}; npos = sum(map(len, gts.values()))
            dets = sorted([(p.probability, f, p.bounding_box) for f, ps in preds.items() for p in ps if p.tag_name == cls], key=lambda x: -x[0])
            used = {f: [False]*len(g) for f, g in gts.items()}; tp = []
            for prob, f, bb in dets:
                ious = [iou((bb.left, bb.top, bb.width, bb.height), (g['left'], g['top'], g['width'], g['height'])) for g in gts[f]]
                j = int(np.argmax(ious)) if ious else -1
                hit = j >= 0 and ious[j] >= 0.5 and not used[f][j]
                if hit: used[f][j] = True
                tp.append(hit)
            tp = np.array(tp, dtype=float); ctp, cfp = np.cumsum(tp), np.cumsum(1-tp)
            rec = ctp/max(npos, 1); prec = ctp/np.maximum(ctp+cfp, 1e-9)
            at50 = [d[0] >= 0.5 for d in dets]; tp50 = tp[np.array(at50, dtype=bool)] if dets else np.array([])
            results.append(dict(cls=cls, gt=npos, AP50=ap_voc(rec, prec) if dets else 0.0,
                                P_at_0_5=tp50.mean() if len(tp50) else 0.0, R_at_0_5=tp50.sum()/max(npos, 1)))
        res = pd.DataFrame(results); print(res.round(3).to_string(index=False)); print('mAP@0.5 =', round(res.AP50.mean(), 3))
        holdout_metrics = dict(mAP50=float(res.AP50.mean()),
                               per_class={r.cls: dict(gt=int(r.gt), AP50=float(r.AP50), precision_at_0_5=float(r.P_at_0_5),
                                                      recall_at_0_5=float(r.R_at_0_5)) for r in res.itertuples()})
    else:
        y_true = [m['tags'][0] for m in hold]; y_pred = [max(preds[m['file']], key=lambda p: p.probability).tag_name for m in hold]
        from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
        print(classification_report(y_true, y_pred)); ConfusionMatrixDisplay.from_predictions(y_true, y_pred, xticks_rotation=45); plt.show()
        labels_ = sorted(set(y_true) | set(y_pred))
        holdout_metrics = dict(report=classification_report(y_true, y_pred, output_dict=True, zero_division=0),
                               confusion_matrix=dict(labels=labels_, matrix=confusion_matrix(y_true, y_pred, labels=labels_).tolist()))

    # Visual spot-check
    fig, axs = plt.subplots(1, 4, figsize=(18, 4))
    for a, m in zip(axs, random.sample(hold, min(4, len(hold)))):
        im = Image.open(IMG_OUT/m['file']); W, H = im.size; a.imshow(im); a.axis('off')
        for r in m['regions']: a.add_patch(patches.Rectangle((r['left']*W, r['top']*H), r['width']*W, r['height']*H, fill=False, ec='lime', lw=2))
        for p in preds[m['file']]:
            if p.probability >= 0.5 and TASK == 'detection':
                b = p.bounding_box; a.add_patch(patches.Rectangle((b.left*W, b.top*H), b.width*W, b.height*H, fill=False, ec='red', ls='--'))
                a.text(b.left*W, b.top*H, f'{p.tag_name} {p.probability:.2f}', color='red', fontsize=7)
        a.set_title('green = ground truth, red = prediction', fontsize=8)
    plt.show()

## 10. Save `metrics.json`
This file is always written. It records the dataset and cleaning stats, and adds the Azure and holdout metrics when section 8 was run.
It is saved to `/content/work/customvision/metrics.json` and copied to Drive.

In [ ]:
import datetime
def _py(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, Path): return str(o)
    raise TypeError(type(o))

metrics = dict(
    created_utc=datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    dataset=dict(source='https://www.scidb.cn/en/detail?dataSetId=3d739ddb4bdc439a9bf7ef550cae48d8',
                 detected_format=FORMAT, task=TASK, raw_images=len(images)),
    quality_checks={k: len(v) for k, v in issues.items()},
    cleaning=dict(class_map=CLASS_MAP, dropped_rare_tags=rare, min_images_per_tag=MIN_IMAGES_PER_TAG),
    split=dict(seed=SEED, holdout_frac=HOLDOUT_FRAC,
               images=clean.drop_duplicates('image').split.value_counts().to_dict(),
               instances_per_class=pd.crosstab(clean.label, clean.split).to_dict(orient='index')),
    azure=None, holdout=None,
)
if RUN_AZURE:
    metrics['azure'] = dict(project_id=str(project.id), iteration_id=str(iteration.id), publish_name=PUBLISH_NAME,
                            domain=domain.name, internal_metrics=azure_metrics)
    metrics['holdout'] = dict(n_images=len(hold), iou_threshold=0.5 if TASK == 'detection' else None, **holdout_metrics)

(OUT/'metrics.json').write_text(json.dumps(metrics, indent=2, default=_py))
if Path('/content/drive/MyDrive').exists(): shutil.copy(OUT/'metrics.json', '/content/drive/MyDrive/metal_defect_metrics.json')
print(json.dumps(metrics, indent=2, default=_py)[:3000])